In [1]:
import sys

from absl import logging
from ferminet.utils import system
from ferminet import base_config
from ferminet import train
from ferminet.configs import atom

# Optional, for also printing training progress to STDOUT.
# If running a script, you can also just use the --alsologtostderr flag.
logging.get_absl_handler().python_handler.stream = sys.stdout
logging.set_verbosity(logging.INFO)


# Define H2 molecule
cfg = base_config.default()
cfg.system.electrons = (1,1)  # (alpha electrons, beta electrons)
cfg.system.molecule = [system.Atom('H', (0, 0, -1)), system.Atom('H', (0, 0, 1))]

# Set training parameters
cfg.batch_size = 4096
cfg.pretrain.iterations = 0
cfg.mcmc.burn_in = 0
cfg.optim.optimizer = 'minsr'


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
evaluate_loss, mcmc_step, sharded_key, data, params, mcmc_width, logabs_network = train.train(cfg, wandb_monitoring=False)

cfg.optim.optimizer = 'minsr'


INFO:absl:Starting QMC with 1 XLA devices per host across 1 hosts.
INFO:absl:No checkpoint found. Training new model.
INFO:absl:Burning in MCMC chain for 0 steps
INFO:absl:Completed burn-in MCMC steps
INFO:absl:Initial energy: -1.1670 E_h


### Constructing Grunter through streaming gradients

In [4]:
import jax
import jax.numpy as jnp
from jax import tree_util
import gc
from tqdm.notebook import tqdm

In [5]:
from ferminet import constants

In [6]:
# squeezing pmap dimension to work on 1 gpu
data_positions = jnp.squeeze(data.positions, axis=0)
data_spins = jnp.squeeze(data.spins, axis=0)
data_atoms = jnp.squeeze(data.atoms, axis=0)
data_charges = jnp.squeeze(data.charges, axis=0)
params_ = jax.tree.map(lambda x: jnp.squeeze(x, axis=0), params)

Define a jacobian vector product here based on params

And then a second one for the second vector X(XT)x

In [7]:
batch_network = jax.vmap(
      logabs_network, in_axes=(None, 0, 0, 0, 0), out_axes=0
  ) # Multi sample network output

grad_params = jax.grad(logabs_network, argnums=0)  # Single sample grad wrt model output

batch_network_grad = jax.vmap(
      grad_params, in_axes=(None, 0, 0, 0, 0), out_axes=0
  )  # Multi sample grad wrt model output (memory problems very quickly)

In [25]:
# Flatten once to capture treedef and shapes
leaves, treedef = jax.tree_util.tree_flatten(params_)
shapes = [leaf.shape for leaf in leaves]
sizes = [leaf.size for leaf in leaves]

def flat_to_pytree(flat_vec):
    idx = 0
    new_leaves = []
    for shape, size in zip(shapes, sizes):
        new_leaves.append(flat_vec[idx:idx + size].reshape(shape))
        idx += size
    return jax.tree_util.tree_unflatten(treedef, new_leaves)


In [47]:
def params_to_array(params_pytree, batched=False):
    leaves, _ = jax.tree_util.tree_flatten(params_pytree)
    if batched:
        batch_size = leaves[0].shape[0]
        flat_leaves = [jnp.reshape(leaf, (batch_size, -1)) for leaf in leaves]
        return jnp.concatenate(flat_leaves, axis=1)  # Shape: (batch_size, n_params)
    else:
        return jnp.concatenate([jnp.ravel(leaf) for leaf in leaves], axis=0)

### First jacfwd product 

In [19]:
def f(p):
    return batch_network(p, data_positions, data_spins, data_atoms, data_charges)

jvp_func = jax.linearize(f, params_)[1]

In [21]:
n_params = sum(jnp.size(p) for p in jax.tree_util.tree_leaves(params))

In [22]:
key = jax.random.PRNGKey(0)
n = n_params
x = jax.random.normal(key, shape=(n,))

In [23]:
len(x)

667104

In [33]:
A = lambda x: jvp_func(flat_to_pytree(x))

## Second jacrev product

In [55]:
jacrev = lambda v: params_to_array(jax.vjp(f, params_)[1](v))

In [56]:
key = jax.random.PRNGKey(0)
n = 4096
x = jax.random.normal(key, shape=(n,))
jacrev(x)

Array([ 0.19293904, -1.8781347 , 11.407321  , ..., 27.623943  ,
       -5.174226  , -9.841717  ], dtype=float32)

In [57]:
monster = lambda v: jacrev(A(v) / 4096)

In [58]:
key = jax.random.PRNGKey(0)
n = n_params
x = jax.random.normal(key, shape=(n,))

t = monster(x)

In [ ]:
(update,) = jax.vjp(f, params_)[1](log_psi_jac_x / batch_size)
if self.config.center_gradients:
    # update = update - g * <g, x>
    innerprod = tree_dot(mean_grads, x)
    update = jax.tree_util.tree_map(lambda u, g: u - g * innerprod, update, mean_grads)
# update = update + damping * x
update = jax.tree_util.tree_map(lambda u, x_: u + damping * x_, update, x)
update = pmean(update)
return update

### Getting b loss grads

In [40]:
loss_and_grad = jax.value_and_grad(evaluate_loss, argnums=0, has_aux=True)

In [44]:
lg = constants.pmap(loss_and_grad)

In [45]:
(loss, aux_data), grad = lg(params, sharded_key, data)

In [36]:
eval_loss = constants.pmap(evaluate_loss)

In [48]:
flat_grads = params_to_array(grad)

In [49]:
flat_grads.shape

(667104,)

In [ ]:
jax.scipy.sparse.linalg.cg(monster, flat_grads,  maxiter=10)

In [31]:
from tqdm.notebook import tqdm

In [ ]:
jvp_func = lambda x: jax.jvp(batch_network, (params_,), (x,))[1]

In [7]:
def params_to_array(params_pytree, batched=False):
    leaves, _ = jax.tree_util.tree_flatten(params_pytree)
    if batched:
        batch_size = leaves[0].shape[0]
        flat_leaves = [jnp.reshape(leaf, (batch_size, -1)) for leaf in leaves]
        return jnp.concatenate(flat_leaves, axis=1)  # Shape: (batch_size, n_params)
    else:
        return jnp.concatenate([jnp.ravel(leaf) for leaf in leaves], axis=0)


In [31]:
def clear_memory():
    jax.clear_caches()
    gc.collect()
    jax.device_put(jax.numpy.zeros(1)).block_until_ready()

In [36]:
grad_params = val_grad(params_, data_positions[0], data_spins[0], data_atoms[0], data_charges[0])

In [43]:
# limits of batched based gradients
n_samples = 64

param_grads = batch_network_grad(
    params_, data_positions[:n_samples], data_spins[:n_samples], data_atoms[:n_samples], data_charges[:n_samples])

In [8]:
def get_batch_network_gradients(pos_start, pos_end):
    return batch_network_grad(
        params_,
        data_positions[pos_start:pos_end], data_spins[pos_start: pos_end], data_atoms[pos_start: pos_end], data_charges[pos_start: pos_end])

In [57]:
def A(x, n_params):
    """ Takes in the vector x (N_samples) and uses it as a premultiplier for the gradients """
    output_vec = jnp.zeros(n_params)

    minibatch = 64
    for i in range(0, n_samples, minibatch):
        pos_start = i * minibatch
        pos_end = i * minibatch + minibatch
        output_vec = output_vec.at[:].add(
            get_batch_network_gradients(pos_start, pos_end) @ x[pos_start:pos_end])

    return output_vec

In [ ]:
n_params = params_to_array(params_).shape[0]
key = jax.random.PRNGKey(0)
n = 4096
x = jax.random.normal(key, shape=(n,))
output_vec = jnp.zeros(n_params)

minibatch = 4
for i in tqdm(range(0, n, minibatch)):
    pos_start = i * minibatch
    pos_end = i * minibatch + minibatch
    output_vec = output_vec.at[:].add(
        params_to_array(get_batch_network_gradients(pos_start, pos_end), batched=True).T @ x[pos_start:pos_end])